In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dhruvgutpa/fine-tuning1001/adapter_model.safetensors
/kaggle/input/datasets/dhruvgutpa/fine-tuning1001/adapter_config.json
/kaggle/input/datasets/dhruvgutpa/fine-tuning1001/README.md
/kaggle/input/datasets/dhruvgutpa/fine-tuning1001/tokenizer.json
/kaggle/input/datasets/dhruvgutpa/fine-tuning1001/tokenizer_config.json
/kaggle/input/datasets/dhruvgutpa/fine-tuning1001/chat_template.jinja


In [2]:
!pip install -q -U unsloth peft transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.2/82.2 MB 23.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 111.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 47.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 94.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 81.3 MB/s eta 0:00:00:00:01
   ━━━

In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [4]:
adapter_path = "/kaggle/input/datasets/dhruvgutpa/fine-tuning1001"

In [5]:
import os

print(os.listdir(adapter_path))

['adapter_model.safetensors', 'adapter_config.json', 'README.md', 'tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja']


In [6]:
from unsloth import FastLanguageModel

base_model = "meta-llama/Llama-3.2-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.12: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [7]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    model,
    adapter_path
)

In [8]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [10]:
messages = [
    {
        "role": "system",
        "content": (
            "You are CyberShield, a defensive cybersecurity "
            "assistant. Provide accurate, ethical, and defensive "
            "cybersecurity guidance."
        )
    },
    {
        "role": "user",
        "content": "What is SQL injection and how can it be prevented?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.3,
    top_p=0.9,
    do_sample=True,
)

response_tokens = outputs[0][inputs.shape[-1]:]

response = tokenizer.decode(
    response_tokens,
    skip_special_tokens=True
)

print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SQL injection (SQLi) is a type of web application vulnerability that allows attackers to inject malicious SQL code into a web application's database, potentially leading to unauthorized data access, modification, or deletion. This occurs when user input is not properly sanitized or parameterized, allowing attackers to inject arbitrary SQL statements into the database query.\\n\\nThe attack typically involves inserting malicious SQL code into a web application's input fields, such as usernames, passwords, or search queries. The injected code is then executed by the database, allowing the attacker to extract sensitive data, modify existing records, or escalate privileges. SQLi can be used for data theft, account takeover, or privilege escalation.\\n\\nPrevention of SQL injection requires a combination of secure coding practices and robust input validation. Here are some key measures:\\n\\n1. **Input Validation**: Implement strict input validation to ensure that all user inputs are saniti

In [1]:
!pip install -q -U gradio unsloth peft transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 60.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.2/82.2 MB 21.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 45.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 2.8 MB/s eta 0:00:00
 

In [2]:
import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

CUDA: True
GPU: Tesla T4
VRAM: 14.56 GB


In [3]:
from unsloth import FastLanguageModel
from peft import PeftModel

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

ADAPTER_PATH = (
    "/kaggle/input/datasets/"
    "dhruvgutpa/fine-tuning1001"
)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

model = PeftModel.from_pretrained(
    model,
    ADAPTER_PATH
)

FastLanguageModel.for_inference(model)

print("✅ CyberShield loaded!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.12: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:310: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:310: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


✅ CyberShield loaded!


In [4]:
import torch
import re

def clean_response(text):
    # Convert literal escaped newlines into real newlines
    text = text.replace("\\n", "\n")
    text = text.replace("\\t", "\t")

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove accidental leading/trailing whitespace
    text = text.strip()

    return text


def generate_response(message, history):

    messages = [
        {
            "role": "system",
            "content": (
                "You are CyberShield, a professional defensive "
                "cybersecurity assistant.\n\n"
                "Answer the user's question directly and clearly.\n"
                "Keep simple questions concise.\n"
                "Use headings and bullet points when useful.\n"
                "Do not unnecessarily repeat information.\n"
                "For simple definitions, answer in 2-5 sentences.\n"
                "For technical questions, provide structured explanations.\n"
                "Focus on defensive, ethical cybersecurity guidance."
            )
        }
    ]

    # Add conversation history
    if history:
        for item in history:

            if isinstance(item, dict):

                role = item.get("role")
                content = item.get("content")

                if role in ["user", "assistant"] and content:
                    messages.append({
                        "role": role,
                        "content": content
                    })

    # Current question
    messages.append({
        "role": "user",
        "content": message
    })

    # Tokenize
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    # Generate
    with torch.no_grad():

        outputs = model.generate(
            input_ids=inputs,

            # Keep answers shorter
            max_new_tokens=256,

            # More deterministic
            temperature=0.2,

            # Better focused responses
            top_p=0.85,

            # Prevent excessive repetition
            repetition_penalty=1.1,

            do_sample=True,

            use_cache=True,
        )

    # Remove input tokens
    response_tokens = outputs[0][inputs.shape[-1]:]

    response = tokenizer.decode(
        response_tokens,
        skip_special_tokens=True
    )

    # Clean model output
    response = clean_response(response)

    return response

In [5]:
import gradio as gr

with gr.Blocks(
    title="CyberShield",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown(
        """
        # 🛡️ CyberShield

        ### Defensive Cybersecurity AI Assistant

        **Llama 3.2 3B Instruct • QLoRA • 4-bit NF4 • Unsloth**

        Ask questions about cybersecurity, threat detection,
        vulnerabilities, incident response, secure coding,
        network security, and defensive security practices.
        """
    )

    chatbot = gr.Chatbot(
        height=600,
        label="CyberShield"
    )

    msg = gr.Textbox(
        placeholder="Ask CyberShield a cybersecurity question...",
        label="Your Question",
        lines=2
    )

    with gr.Row():

        submit = gr.Button(
            "🛡️ Ask CyberShield",
            variant="primary"
        )

        clear = gr.Button(
            "🗑️ Clear"
        )

    gr.Markdown(
        """
        ---
        **Model:** Llama 3.2 3B Instruct  
        **Fine-tuning:** QLoRA  
        **Dataset:** 8,000 cybersecurity examples  
        **Training:** 7,200 examples  
        **Validation:** 800 examples  
        **Inference:** Kaggle GPU
        """
    )

    def respond(message, history):

        if not message.strip():
            return "", history

        response = generate_response(
            message,
            history
        )

        history = history + [
            {
                "role": "user",
                "content": message
            },
            {
                "role": "assistant",
                "content": response
            }
        ]

        return "", history

    submit.click(
        respond,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot]
    )

    msg.submit(
        respond,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot]
    )

    clear.click(
        lambda: [],
        outputs=chatbot
    )

print("✅ Gradio app created!")

/tmp/ipykernel_58/1230405067.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


✅ Gradio app created!


In [6]:
demo.launch(
    share=True,
    debug=False
)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://05f712f33c9fcd202c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
